In [2]:
# Install the libraries used in this notebook.
!pip install pandas requests folium python-dotenv

In [26]:
# Import the libraries and load the TomTom API key.

import os

import pandas as pd
import requests
import folium

from dotenv import load_dotenv

load_dotenv()
TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY")


# Define the NYC locations to analyze.
NYC_LOCATIONS = [
    {"name": "Times Square", "lat": 40.7580, "lng": -73.9855},
    {"name": "Central Park South", "lat": 40.7651, "lng": -73.9769},
    {"name": "Grand Central", "lat": 40.7527, "lng": -73.9772},
    {"name": "Wall Street", "lat": 40.7066, "lng": -74.0090},
    {"name": "Brooklyn Bridge", "lat": 40.7061, "lng": -73.9969},
]

In [27]:
# Retrieve live traffic data for a location from the TomTom API.
def get_flow(lat, lng):
    url = (
        "https://api.tomtom.com/traffic/services/4/"
        f"flowSegmentData/relative0/10/json?point={lat},{lng}&key={TOMTOM_API_KEY}"
    )
    return requests.get(url, timeout=10).json().get("flowSegmentData")  # dict or None


In [28]:
# Collect traffic data for each NYC location.
rows = []
for p in NYC_LOCATIONS: 
    d = get_flow(p["lat"], p["lng"])
    if not d:
        print(f"No data for {p['name']}")
        continue
    rows.append({
        "name": p["name"],
        "lat": p["lat"],
        "lng": p["lng"],
        "currentSpeed": d.get("currentSpeed"),
        "freeFlowSpeed": d.get("freeFlowSpeed"),
        "currentTravelTime": d.get("currentTravelTime"),
        "freeFlowTravelTime": d.get("freeFlowTravelTime"),
        "confidence": d.get("confidence"),
    })

df = pd.DataFrame(rows)
df


,name,lat,lng,currentSpeed,freeFlowSpeed,currentTravelTime,freeFlowTravelTime,confidence
0,Times Square,40.7580,-73.9855,5,9,286,159,1.000000
1,Central Park South,40.7651,-73.9769,11,11,147,147,1.000000
2,Grand Central,40.7527,-73.9772,4,11,1032,375,1.000000
3,Wall Street,40.7066,-74.0090,8,17,403,189,0.995751
4,Brooklyn Bridge,40.7061,-73.9969,15,42,567,202,1.000000


In [29]:
# Calculate the congestion ratio for each location.
df["congestion_ratio"] = df["currentSpeed"] / df["freeFlowSpeed"]
df


,name,lat,lng,currentSpeed,freeFlowSpeed,currentTravelTime,freeFlowTravelTime,confidence,congestion_ratio
0,Times Square,40.7580,-73.9855,5,9,286,159,1.000000,0.555556
1,Central Park South,40.7651,-73.9769,11,11,147,147,1.000000,1.000000
2,Grand Central,40.7527,-73.9772,4,11,1032,375,1.000000,0.363636
3,Wall Street,40.7066,-74.0090,8,17,403,189,0.995751,0.470588
4,Brooklyn Bridge,40.7061,-73.9969,15,42,567,202,1.000000,0.357143


In [30]:
!pip install folium


In [31]:
# Create an interactive map of the traffic data.
import folium

m = folium.Map(location=[40.733, -73.995], zoom_start=12)

for _, r in df.iterrows():
    folium.Marker(
        location=[r["lat"], r["lng"]],
        popup=f"{r['name']}: {r['currentSpeed']} (free {r['freeFlowSpeed']})",
    ).add_to(m)

m  # shows in notebook


In [36]:
# Calculate a traffic risk score.
df["traffic_risk"] = (1 - df["congestion_ratio"]) * 100
city_risk = df["traffic_risk"].mean()

print(df[["name", "congestion_ratio", "traffic_risk"]])
print("\nAverage city traffic risk:", round(city_risk, 1))


                 name  congestion_ratio  traffic_risk
0        Times Square          0.555556     44.444444
1  Central Park South          1.000000      0.000000
2       Grand Central          0.363636     63.636364
3         Wall Street          0.470588     52.941176
4     Brooklyn Bridge          0.357143     64.285714

Average city traffic risk: 45.1


In [25]:
# Create a color-coded map based on the traffic risk score.
import folium

m = folium.Map(location=[40.733, -73.995], zoom_start=12)

for _, r in df.iterrows():
    risk = r["traffic_risk"]
    if risk < 30:
        color = "green"
    elif risk < 60:
        color = "orange"
    else:
        color = "red"
    
    folium.CircleMarker(
        location=[r["lat"], r["lng"]],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"{r['name']} - Risk: {round(risk,1)}",
    ).add_to(m)

m
